# Train deepfake-detect on Kaggle

Trains **v0.2**: CLIP with only its LayerNorms tuned, plus a linear head, on **FaceForensics++ (c23)** face crops with self-blended fakes, using the official train/val/test splits. Then it tests on **Celeb-DF v2**, a dataset the model never sees during training. That second number is the honest one. (v0.1's frozen linear probe is `python -m deepfake_detect.train` with the same arguments.)

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (one GPU is used). Turn **Internet on**.
2. Both datasets need an access request to their authors ([FaceForensics++](https://github.com/ondyari/FaceForensics), [Celeb-DF](https://github.com/yuezunli/celeb-deepfakeforensics)). Their licences forbid redistribution, so upload your copy as a **private** Kaggle dataset and attach it with *Add Input*.
3. Attach both datasets with *Add Input*. The notebook finds them under `/kaggle/input/` by itself.

Expected runtime: ~2–3 h: ~1 h extracting face crops on all CPU cores (cached in `crops/` for re-runs), then ~1 h of training and evaluation on the GPU.

In [1]:
REPO = "https://github.com/CodeRafay/deepfake-detect"
BRANCH = "main"
!rm -rf /kaggle/working/deepfake-detect
!git clone -q -b $BRANCH $REPO /kaggle/working/deepfake-detect
!pip install -q /kaggle/working/deepfake-detect pytest
!python -m pytest -q /kaggle/working/deepfake-detect/tests

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
.......                                                                  [100%]
7 passed in 8.45s


In [2]:
import json, urllib.request
from pathlib import Path

def find_input(name):
    """Kaggle mounts inputs at varying depths (e.g. /kaggle/input/datasets/<user>/<name>/), so search for a marker."""
    hits = sorted(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"no '{name}' under /kaggle/input - is the dataset attached via Add Input?")
    return hits[0].parent

FFPP = find_input("original_sequences")              # folder holding original_sequences/ and manipulated_sequences/
CELEBDF = find_input("List_of_testing_videos.txt")   # folder holding Celeb-real/, Celeb-synthesis/, YouTube-real/
print("FF++:", FFPP)
print("Celeb-DF:", CELEBDF)
METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
DATA, CDF = Path("/kaggle/working/data"), Path("/kaggle/working/celebdf")

def link(src, dst):
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(src)

# Official FF++ splits are by source-video pair, so no identity appears in two splits.
for split in ["train", "val", "test"]:
    url = f"https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{split}.json"
    for a, b in json.load(urllib.request.urlopen(url)):
        for v in (a, b):
            link(FFPP / f"original_sequences/youtube/c23/videos/{v}.mp4", DATA / split / "real" / f"{v}.mp4")
        for m in METHODS:
            for x, y in ((a, b), (b, a)):
                link(FFPP / f"manipulated_sequences/{m}/c23/videos/{x}_{y}.mp4", DATA / split / "fake" / f"{m}_{x}_{y}.mp4")

# Celeb-DF official test list: "1 path" = real, "0 path" = fake.
for line in (CELEBDF / "List_of_testing_videos.txt").read_text().splitlines():
    if line.strip():
        lab, rel = line.split()
        link(CELEBDF / rel, CDF / ("real" if lab == "1" else "fake") / rel.replace("/", "_"))

for d in [DATA / s / c for s in ["train", "val", "test"] for c in ["real", "fake"]] + [CDF / "real", CDF / "fake"]:
    print(f"{d}: {len(list(d.glob('*')))} videos")

FF++: /kaggle/input/notebooks/rafayadeel/ffpp-c23-download/ffpp
Celeb-DF: /kaggle/input/datasets/rafayadeel/celeb-df-v2
/kaggle/working/data/train/real: 720 videos
/kaggle/working/data/train/fake: 2880 videos
/kaggle/working/data/val/real: 140 videos
/kaggle/working/data/val/fake: 560 videos
/kaggle/working/data/test/real: 140 videos
/kaggle/working/data/test/fake: 560 videos
/kaggle/working/celebdf/real: 178 videos
/kaggle/working/celebdf/fake: 340 videos


In [3]:
!cd /kaggle/working && python -m deepfake_detect.finetune --data data --eval celebdf=celebdf --out head.safetensors

crops train: 100%|████████████████████████| 3600/3600 [1:31:50<00:00,  1.53s/it]
data/train: 28752 face crops from 3600 files
crops val: 100%|██████████████████████████████| 700/700 [16:38<00:00,  1.43s/it]
data/val: 5590 face crops from 700 files
crops test: 100%|█████████████████████████████| 700/700 [17:32<00:00,  1.50s/it]
data/test: 5599 face crops from 700 files
crops celebdf: 100%|██████████████████████████| 518/518 [01:42<00:00,  5.04it/s]
celebdf: 4139 face crops from 518 files
preprocessor_config.json: 100%|████████████████| 316/316 [00:00<00:00, 1.46MB/s]
config.json: 100%|█████████████████████████| 4.52k/4.52k [00:00<00:00, 11.2MB/s]

model.safetensors: downloading bytes: ██▍                   |  193MB, 14.1MB/s  
model.safetensors: reconstructing file:   4%|▏     | 67.1MB / 1.71GB            
model.safetensors: downloading bytes: ██████▏               |  477MB, 39.4MB/s  
model.safetensors: downloading bytes: ███████▋              |  597MB, 50.0MB/s  
model.safetensors: re

Paste the table above into the README's **Results** section. Report it as is, even if the Celeb-DF number is unflattering. Next, sanity-check a few test videos:

In [4]:
from deepfake_detect import Detector
det = Detector("/kaggle/working/head.safetensors")
for p in sorted((CDF / "real").glob("*"))[:3] + sorted((CDF / "fake").glob("*"))[:3]:
    r = det.predict(p)
    print(p.parent.name, r["label"], r["score"])

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{

real real 0.3237
real real 0.3543
real fake 0.6283
fake fake 0.9679
fake fake 0.9259
fake fake 0.9758


## Publish this version to Hugging Face
One-time setup: create a **Write** token at huggingface.co → Settings → Access Tokens, then add it to this notebook under *Add-ons → Secrets* with the label `HF_TOKEN`. Never paste the token into a cell.

Set `VERSION` **before** running the notebook: `v<N>-<what changed>`, e.g. `v0.2-clip-lntune-sbi`. The head and metrics are uploaded into that folder. Existing versions are never overwritten. Afterwards, download the files and this executed notebook, and commit them to `models/<VERSION>/` on GitHub (as `training_run.ipynb`), plus a row in `models/README.md`.

In [5]:
VERSION = "v0.2-clip-lntune-sbi"        # v<N>-<what changed>; never reuse a published name
HF_REPO = "CodeRafay/deepfake-detect"    # must match MODELS_REPO in deepfake_detect/model.py

from huggingface_hub import CommitOperationAdd, HfApi
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None

if "CHANGE-ME" in VERSION:
    print("VERSION not set: skipping upload. head.safetensors is in this notebook's output.")
elif not token:
    print("No HF_TOKEN secret attached: skipping upload. head.safetensors is in this notebook's output.")
else:
    api = HfApi(token=token)
    if api.file_exists(HF_REPO, f"{VERSION}/head.safetensors"):
        print(f"{VERSION} already exists on Hugging Face and versions are never replaced. Pick a new VERSION.")
    else:
        api.create_commit(HF_REPO, commit_message=f"add {VERSION}", operations=[
            CommitOperationAdd(f"{VERSION}/{f}", f"/kaggle/working/{f}") for f in ["head.safetensors", "head.metrics.json"]])
        print(f"done: https://huggingface.co/{HF_REPO}/tree/main/{VERSION}")

No HF_TOKEN secret attached: skipping upload. head.safetensors is in this notebook's output.
